# Baseline correction

It is important to apply normalization to the data based on the baseline pupil size. More details can be found in [API reference](../../../api.rst) or in Mathôt et al. (2018).

In [ ]:
import matplotlib.pyplot as plt
import piprkit

## Load a sample measurement

We load some data.

In [ ]:
pm = piprkit.load_simulated_pupil()

fig, ax = plt.subplots(1, 1, figsize=(8, 2), sharex=True)
pm.plot(ax=ax, label='Original Data', color='gray')
pm.plot_light_stimulus(ax=ax, color='yellow', alpha=0.3)
ax.legend()
plt.show()

## Compute and apply baseline correction

The `baseline_duration` can be chosen; it defines over what period before the light stimulus the baseline size is averaged. The baseline is either retrieved through `metrics.baseline` or applied using `filters.apply_baseline_correction`.

In [ ]:
# Calculate the baseline, and apply the baseline correction. Note. This requires the light stimulus to be set.

baseline_duration = 3 # seconds, counting back from light onset

baseline = pm.metrics.baseline(duration=baseline_duration)
baseline_pm = pm.filters.apply_baseline_correction(duration=baseline_duration, inplace=False)

fig, ax = plt.subplots(2, 1, figsize=(8, 4), sharex=True)
pm.plot(ax=ax[0], label='Original Data', color='gray', alpha=0.5)
baseline_pm.plot(ax=ax[1], label='Baseline Corrected Data', color='blue')
pm.plot_light_stimulus(ax=ax[0], color='yellow', alpha=0.3)
baseline_pm.plot_light_stimulus(ax=ax[1], color='yellow', alpha=0.3)

baseline.plot(ax=ax[0], color="red", linestyle="--", label=f"{baseline.get_label()}: {baseline:.2f}")

# The new baseline should be at exactly 1.00 (100%)
new_baseline = baseline_pm.metrics.baseline(duration=baseline_duration)
new_baseline.plot(ax=ax[1], color="red", linestyle="--", label=f"{new_baseline.get_label()}: {new_baseline:.2f}")

ax[0].legend()
ax[1].legend()
plt.show()

## Baseline correction methods

The baseline takes a `method` argument, which defaults to `ratio`. Other options are also available.

In [ ]:
methods = ["none", "ratio", "subtractive", "percent"]

fig, ax = plt.subplots(len(methods), 1, figsize=(8, 8), sharex=True)
for axis, method in zip(ax, methods):
    if method == "none":
        corrected = pm
    else:
        corrected = pm.filters.apply_baseline_correction(duration=3, method=method, inplace=False)
    pipr = corrected.metrics.pipr_6s()
    auc = corrected.metrics.auc_early(baseline_duration=3)
    print(f"{str(corrected.get_baseline_correction()):<35}{pipr.get_name():<21}{pipr:8.3f}   {auc.get_name():<21}{auc:8.3f}")
    corrected.plot(ax=axis, color="blue", label=method)
    corrected.plot_light_stimulus(ax=axis, color="yellow", alpha=0.3)
    pipr.plot(ax=axis, color="red")
    auc.plot(ax=axis, color="green")
    axis.legend()
plt.show()